In [ ]:
# ==============================================================================
# CELDA 0: RUTAS DEL REPOSITORIO
# Este cuaderno leia sus datos desde Google Drive. Ahora los lee del propio
# repositorio, de modo que corre en cualquier clon sin configuracion previa.
# La raiz se resuelve buscando hacia arriba: funciona igual si el cuaderno se
# ejecuta desde su carpeta o desde la raiz del proyecto.
# ==============================================================================
from pathlib import Path

def _raiz_del_repositorio() -> Path:
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / 'pyproject.toml').exists() and (carpeta / 'quanta').is_dir():
            return carpeta
    raise RuntimeError(
        'No se encontro la raiz del repositorio. Ejecute el cuaderno desde '
        'dentro del proyecto indice-sned.'
    )

RAIZ            = _raiz_del_repositorio()
RUTA_RAW        = (RAIZ / 'data' / 'raw').as_posix() + '/'
RUTA_PROCESADOS = (RAIZ / 'data' / 'processed').as_posix() + '/'
RUTA_REGISTRO   = (RAIZ / 'models' / 'registry').as_posix() + '/'
RUTA_METADATOS  = (RAIZ / 'models' / 'metadata').as_posix() + '/'

print('Raiz del repositorio:', RAIZ)


In [16]:
# ==============================================================================
# CELDA 1: INGESTA MEDIACIONES (CSV, nombres reales confirmados)
# ==============================================================================
import pandas as pd

RUTA = RUTA_RAW + 'mediaciones/'

archivos = {
    2016: '20161231_MEDIACIONES_2016_20200817_PUBL.csv',
    2017: '20171231_MEDIACIONES_2017_20200817_PUBL.csv',
}

med_brutas = {}
for anio, nombre in archivos.items():
    ruta = RUTA + nombre
    for enc in ['utf-8-sig', 'latin-1', 'cp1252']:
        try:
            med_brutas[anio] = pd.read_csv(ruta, encoding=enc, sep=';', low_memory=False)
            break
        except UnicodeDecodeError:
            continue
    print(f"OK | {anio} | {med_brutas[anio].shape[0]} filas x {med_brutas[anio].shape[1]} columnas")

print("\nColumnas 2016:", med_brutas[2016].columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK | 2016 | 423 filas x 25 columnas
OK | 2017 | 1479 filas x 25 columnas

Columnas 2016: ['AGNO', 'MED_MID', 'MED_CANAL', 'MED_ESTADO', 'MED_ESTADO_DET', 'MED_FEC_CREACION', 'MED_MES_CREACION', 'MED_TRIMESTRE_CREACION', 'MED_FEC_CIERRE', 'MED_OFICINA', 'MED_TEMA_INGRESO', 'MED_TEMA_MEDIADO', 'SOL_MRUN', 'EE_RBD_RSIE', 'EE_NOMBRE', 'EE_COD_REGION', 'EE_COD_PROVINCIA', 'EE_COD_COMUNA', 'EE_NOM_COMUNA', 'EE_COD_DEPE', 'EE_DEPE_SIE', 'MED_COD_ENSE2', 'MED_DENUNCIA', 'DEN_MID', 'MED_EFECTIVAS']


In [17]:
# ==============================================================================
# CELDA 2: AGREGACIÓN POR RBD
# ==============================================================================
med_todas = pd.concat(med_brutas.values(), ignore_index=True)

med_todas['EE_RBD_RSIE'] = pd.to_numeric(med_todas['EE_RBD_RSIE'], errors='coerce')
med_todas = med_todas.dropna(subset=['EE_RBD_RSIE'])
med_todas['rbd'] = med_todas['EE_RBD_RSIE'].astype('Int64').astype(str)

agg = med_todas.groupby('rbd').agg(
    mediaciones_total=('MED_MID', 'count'),
    mediaciones_efectivas=('MED_EFECTIVAS', lambda x: (x == 1).sum()),
    mediaciones_de_denuncia=('MED_DENUNCIA', lambda x: (x == 1).sum()),
).reset_index()

print(f"Colegios con al menos 1 mediación: {len(agg)}")
print(agg.describe())

RUTA_SALIDA = RUTA_PROCESADOS
agg.to_parquet(RUTA_SALIDA + 'mediaciones_2016_17_por_rbd.parquet', index=False)
print("Guardado OK")

Colegios con al menos 1 mediación: 1317
       mediaciones_total  mediaciones_efectivas  mediaciones_de_denuncia
count        1317.000000            1317.000000              1317.000000
mean            1.411541               0.887623                 0.066059
std             0.841074               0.841265                 0.257491
min             1.000000               0.000000                 0.000000
25%             1.000000               0.000000                 0.000000
50%             1.000000               1.000000                 0.000000
75%             2.000000               1.000000                 0.000000
max             8.000000               7.000000                 2.000000
Guardado OK


In [18]:
# ==============================================================================
# CELDA 3: INTEGRAR MEDIACIONES A LA TABLA DE ENTRENAMIENTO
# ==============================================================================
RUTA = RUTA_PROCESADOS
df_modelo = pd.read_parquet(RUTA + 'tabla_modelo_final_v5.parquet')
med = pd.read_parquet(RUTA + 'mediaciones_2016_17_por_rbd.parquet')

df_modelo_v6 = pd.merge(df_modelo, med, on='rbd', how='left', validate='one_to_one')

cols_med = [c for c in med.columns if c != 'rbd']
df_modelo_v6[cols_med] = df_modelo_v6[cols_med].fillna(0)

print(f"Filas: {len(df_modelo_v6)} (antes: {len(df_modelo)})")
print(f"Con al menos 1 mediación: {(df_modelo_v6['mediaciones_total']>0).sum()}")

df_modelo_v6.to_parquet(RUTA + 'tabla_modelo_final_v6.parquet', index=False)
print(df_modelo_v6.shape)

Filas: 7754 (antes: 7754)
Con al menos 1 mediación: 1074
(7754, 51)


In [22]:
print("Columnas 2018:", med_brutas_v2[2018].columns.tolist())
print("\nColumnas 2022:", med_brutas_v2[2022].columns.tolist())

Columnas 2018: ['AGNO', 'MED_MID', 'MED_CANAL', 'MED_ESTADO', 'MED_ESTADO_DET', 'MED_FEC_CREACION', 'MED_MES_CREACION', 'MED_TRIMESTRE_CREACION', 'MED_FEC_CIERRE', 'MED_OFICINA', 'MED_TEMA_INGRESO', 'MED_TEMA_MEDIADO', 'SOL_MRUN', 'RBD', 'RSIE', 'EE_NOMBRE', 'EE_COD_REGION', 'EE_COD_PROVINCIA', 'EE_COD_COMUNA', 'EE_NOM_COMUNA', 'EE_COD_DEPE', 'EE_DEPE_SIE', 'MED_COD_ENSE2', 'MED_DENUNCIA', 'DEN_MID', 'MED_EFECTIVAS']

Columnas 2022: ['AGNO', 'MED_MID', 'MED_CANAL', 'MED_ESTADO', 'MED_ESTADO_DET', 'MED_FEC_CREACION', 'MED_MES_CREACION', 'MED_TRIMESTRE_CREACION', 'MED_FEC_CIERRE', 'MED_OFICINA', 'MED_CANAL_SESIONES', 'MED_TEMA_INGRESO', 'MED_TEMA_MEDIADO', 'SOL_MRUN', 'SOL_GENERO', 'SOL_SEXO', 'AFEC_MRUN', 'AFEC_GENERO', 'AFEC_SEXO', 'RBD', 'RSIE', 'EE_NOMBRE', 'EE_COD_REGION', 'EE_COD_PROVINCIA', 'EE_COD_COMUNA', 'EE_NOM_COMUNA', 'EE_COD_DEPE', 'EE_DEPE_SIE', 'MED_COD_ENSE2', 'MED_DENUNCIA', 'DEN_MID', 'MED_EFECTIVAS']


In [23]:
# ==============================================================================
# CELDA 4 (NOTEBOOK MEDIACIONES): 2018-2022 (corregida: usa 'RBD' directo)
# ==============================================================================
med_todas_v2 = pd.concat(med_brutas_v2.values(), ignore_index=True)
med_todas_v2['RBD'] = pd.to_numeric(med_todas_v2['RBD'], errors='coerce')
med_todas_v2 = med_todas_v2.dropna(subset=['RBD'])
med_todas_v2['rbd'] = med_todas_v2['RBD'].astype('Int64').astype(str)

agg_v2 = med_todas_v2.groupby('rbd').agg(
    mediaciones_total=('MED_MID', 'count'),
    mediaciones_efectivas=('MED_EFECTIVAS', lambda x: (x == 1).sum()),
    mediaciones_de_denuncia=('MED_DENUNCIA', lambda x: (x == 1).sum()),
).reset_index()

print(f"Colegios con mediación (2018-22): {len(agg_v2)}")
RUTA_SALIDA = RUTA_PROCESADOS
agg_v2.to_parquet(RUTA_SALIDA + 'mediaciones_2018_22_por_rbd.parquet', index=False)
print("Guardado OK")

Colegios con mediación (2018-22): 2654
Guardado OK
